torch - deep learning library 
numpy - array operations    
cv2 - image processing
matplotlib - display images 
os - interact with files 
glob - finds files in the folders

saved trained ai model - fusion_model.pth
image will be resizes to 128x28 pixels 

the hardware configuration - checks whether cuda is available else just go with the cpu version 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import glob

# --- CONFIGURATION ---
MODEL_PATH = "fusion_model.pth" 
IMG_SIZE = 256               

# --- HARDWARE SETUP: NATIVE CUDA ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Generator hardware: {DEVICE}")

class FusionNet (nn.Model) - this is an ai architechure that creates CNN
there are three sections to these layers encoder , bottleneck layer , decoder

encoder - take images , extract images , shrink spacial size and increase depth , there will be six input channels 3 channels from image 1 and 3 channels from image 2 (R,G,B)

bottleneck layer - this is a deep thinking layer . it processes compressed information to figure out motion between frames 

decoder layer - this upscales the image. it takes compressed data back into reconstructed image .

final layer - the output is three channels (RGB image)

def foreward - this defies how a data showld flow through the network 
flow 
imput -> decoder -> bottleneck -> decoder -> output image   

In [ ]:
# --- 1. DEFINE THE MODEL ARCHITECTURE ---
class FusionNet(nn.Module):
    def __init__(self):
        super(FusionNet, self).__init__()
        self.enc1 = nn.Conv2d(6, 32, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2) 
        self.enc3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.bottle = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.dec3 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.final = nn.Conv2d(32, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        b = F.relu(self.bottle(e3))
        d3 = F.relu(self.dec3(b))
        d2 = F.relu(self.dec2(d3))
        return torch.sigmoid(self.final(d2))

model - FusionNet....... - this creates the neural netwrk 
then it loads the saved training weights . this means that ai remembers what it learned during the training 
model.eval() - sets the model to inference mode , not training mode 

In [ ]:
model = FusionNet().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval() 
    print("✅ AI Brain Loaded Successfully!")
else:
    print(f"❌ Error: {MODEL_PATH} not found. Train the model first.")

here you have the image warping function and generate intermediate frame function 
warping function 
    this part uses optical flow , ie how does the pixels move between each frames . then it moves the pixel accrgin to that motion it uses F.grid_sample()

In [ ]:
# --- 3. HELPER FUNCTIONS ---
def warp_tensor(img_tensor, flow_tensor):
    N, C, H, W = img_tensor.shape
    xx = torch.arange(0, W).view(1, -1).repeat(H, 1)
    yy = torch.arange(0, H).view(-1, 1).repeat(1, W)
    xx = xx.view(1, 1, H, W).repeat(N, 1, 1, 1)
    yy = yy.view(1, 1, H, W).repeat(N, 1, 1, 1)
    grid = torch.cat((xx, yy), 1).float()
    if img_tensor.device.type != 'cpu': 
        grid = grid.to(DEVICE)
    vgrid = grid + flow_tensor
    vgrid[:, 0, :, :] = 2.0 * vgrid[:, 0, :, :] / max(W - 1, 1) - 1.0
    vgrid[:, 1, :, :] = 2.0 * vgrid[:, 1, :, :] / max(H - 1, 1) - 1.0
    vgrid = vgrid.permute(0, 2, 3, 1)
    return F.grid_sample(img_tensor, vgrid, align_corners=True)

def generate_intermediate(frame1_path, frame2_path):
    print("   -> Reading images...")
    img1 = cv2.imread(frame1_path)
    img2 = cv2.imread(frame2_path)
    
    if img1 is None or img2 is None:
        print("   ❌ Error reading input images.")
        return None, None, None

    img1_s = cv2.resize(img1, (IMG_SIZE, IMG_SIZE))
    img2_s = cv2.resize(img2, (IMG_SIZE, IMG_SIZE))
    
    print("   -> Calculating Optical Flow...")
    prev_gray = cv2.cvtColor(img1_s, cv2.COLOR_BGR2GRAY)
    next_gray = cv2.cvtColor(img2_s, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 0.5, 5, 25, 10, 5, 1.2, 0)
    
    t1 = torch.from_numpy(img1_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t2 = torch.from_numpy(img2_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t_flow = torch.from_numpy(flow).permute(2, 0, 1).float().unsqueeze(0)
    
    t1, t2, t_flow = t1.to(DEVICE), t2.to(DEVICE), t_flow.to(DEVICE)
    
    print("   -> Running CNN Fusion...")
    with torch.no_grad():
        warped_1 = warp_tensor(t1, t_flow * 0.5)
        warped_2 = warp_tensor(t2, t_flow * -0.5)
        cnn_input = torch.cat([warped_1, warped_2], dim=1)
        output = model(cnn_input)
        
    out_img = output.squeeze(0).permute(1, 2, 0).cpu().numpy()
    out_img = (out_img * 255).astype(np.uint8)
    
    return img1_s, out_img, img2_s

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
MODEL_PATH = "fusion_model.pth" 
IMG_SIZE = 256                

# --- HARDWARE SETUP ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Generator hardware: {DEVICE}")

# --- 1. DEFINE THE MODEL ARCHITECTURE ---
class FusionNet(nn.Module):
    def __init__(self):
        super(FusionNet, self).__init__()
        self.enc1 = nn.Conv2d(6, 32, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2) 
        self.enc3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.bottle = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.dec3 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.final = nn.Conv2d(32, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        b = F.relu(self.bottle(e3))
        d3 = F.relu(self.dec3(b))
        d2 = F.relu(self.dec2(d3))
        return torch.sigmoid(self.final(d2))

# --- 2. LOAD THE TRAINED MODEL ---
model = FusionNet().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval() 
    print("✅ AI Brain Loaded Successfully!")
else:
    print(f"❌ Error: {MODEL_PATH} not found. Train the model first.")

# --- 3. CORE WARPING & FUSION LOGIC ---
def warp_tensor(img_tensor, flow_tensor):
    N, C, H, W = img_tensor.shape
    xx = torch.arange(0, W).view(1, -1).repeat(H, 1)
    yy = torch.arange(0, H).view(-1, 1).repeat(1, W)
    xx = xx.view(1, 1, H, W).repeat(N, 1, 1, 1)
    yy = yy.view(1, 1, H, W).repeat(N, 1, 1, 1)
    grid = torch.cat((xx, yy), 1).float()
    if img_tensor.device.type != 'cpu': 
        grid = grid.to(DEVICE)
    vgrid = grid + flow_tensor
    vgrid[:, 0, :, :] = 2.0 * vgrid[:, 0, :, :] / max(W - 1, 1) - 1.0
    vgrid[:, 1, :, :] = 2.0 * vgrid[:, 1, :, :] / max(H - 1, 1) - 1.0
    vgrid = vgrid.permute(0, 2, 3, 1)
    return F.grid_sample(img_tensor, vgrid, align_corners=True)

def interpolate_two_images(img1_s, img2_s):
    """Core function: Takes two numpy images and returns the exactly middle generated image."""
    prev_gray = cv2.cvtColor(img1_s, cv2.COLOR_BGR2GRAY)
    next_gray = cv2.cvtColor(img2_s, cv2.COLOR_BGR2GRAY)
    
    # Calculate Flow
    flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 0.5, 5, 25, 10, 5, 1.2, 0)
    
    # Prepare Tensors
    t1 = torch.from_numpy(img1_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t2 = torch.from_numpy(img2_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t_flow = torch.from_numpy(flow).permute(2, 0, 1).float().unsqueeze(0)
    
    t1, t2, t_flow = t1.to(DEVICE), t2.to(DEVICE), t_flow.to(DEVICE)
    
    # Run AI
    with torch.no_grad():
        warped_1 = warp_tensor(t1, t_flow * 0.5)
        warped_2 = warp_tensor(t2, t_flow * -0.5)
        cnn_input = torch.cat([warped_1, warped_2], dim=1)
        output = model(cnn_input)
        
    out_img = output.squeeze(0).permute(1, 2, 0).cpu().numpy()
    out_img = (out_img * 255).astype(np.uint8)
    
    return out_img

# --- 4. RECURSIVE GENERATION ENGINE ---
def recursive_generation(img1, img2, depth):
    """
    Recursively generates frames.
    Depth 1 = 1 frame
    Depth 2 = 3 frames
    Depth 3 = 7 frames
    """
    if depth == 0:
        return []
    
    # 1. Find the middle frame
    middle_img = interpolate_two_images(img1, img2)
    
    # 2. Recursively find frames on the left and right
    left_frames = recursive_generation(img1, middle_img, depth - 1)
    right_frames = recursive_generation(middle_img, img2, depth - 1)
    
    # 3. Stitch them together in order
    return left_frames + [middle_img] + right_frames

# --- 5. INTERACTIVE MULTI-FRAME EXECUTION ---
print("\n--- Multi-Frame Interpolation Engine ---")

frame_A_path = input("Enter path to FIRST keyframe: ").strip()
frame_B_path = input("Enter path to SECOND keyframe: ").strip()

# Ask for the recursion depth
print("\nHow many frames do you want to generate?")
print(" Type 1 for 1 frame  (Fastest)")
print(" Type 2 for 3 frames (Standard)")
print(" Type 3 for 7 frames (Slower, very smooth)")
depth_input = input("Enter depth (1, 2, or 3): ").strip()
depth = int(depth_input) if depth_input in ['1', '2', '3'] else 1

output_folder = "generated_sequence"
os.makedirs(output_folder, exist_ok=True)

if os.path.exists(frame_A_path) and os.path.exists(frame_B_path):
    print(f"\n⏳ Processing Recursion Depth {depth}...")
    
    # Load and resize the keyframes
    imgA = cv2.resize(cv2.imread(frame_A_path), (IMG_SIZE, IMG_SIZE))
    imgB = cv2.resize(cv2.imread(frame_B_path), (IMG_SIZE, IMG_SIZE))
    
    # Generate all intermediate frames
    generated_frames = recursive_generation(imgA, imgB, depth)
    
    # Save everything to the hard drive in order!
    print(f"💾 Saving {len(generated_frames)} generated frames to '{output_folder}/'...")
    
    # Save Frame A
    cv2.imwrite(os.path.join(output_folder, "frame_00.png"), imgA)
    
    # Save Intermediate Frames
    for i, frame in enumerate(generated_frames):
        # We format the number so it sorts properly (01, 02, 03...)
        filename = f"frame_{i+1:02d}.png" 
        cv2.imwrite(os.path.join(output_folder, filename), frame)
        
    # Save Frame B
    final_index = len(generated_frames) + 1
    cv2.imwrite(os.path.join(output_folder, f"frame_{final_index:02d}.png"), imgB)
    
    print("\n✅ Sequence Generation Complete!")
    print(f"Check the '{output_folder}' folder. The images are numbered in chronological order.")

else:
    print("\n❌ Error: Could not find your input files.")

🚀 Generator hardware: cuda
✅ AI Brain Loaded Successfully!

--- Multi-Frame Interpolation Engine ---


C:\Users\cempl\AppData\Local\Temp\ipykernel_12364\45889547.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DEV


How many frames do you want to generate?
 Type 1 for 1 frame  (Fastest)
 Type 2 for 3 frames (Standard)
 Type 3 for 7 frames (Slower, very smooth)

⏳ Processing Recursion Depth 2...
💾 Saving 3 generated frames to 'generated_sequence/'...

✅ Sequence Generation Complete!
Check the 'generated_sequence' folder. The images are numbered in chronological order.
